# TinyImageNet Training — 4 PE × 3 Seeds

This notebook trains ViT-Base models on **TinyImageNet** for all four PE
strategies (Learned, Sinusoidal, RoPE, ALiBi) for the TPAMI revision.

**Why TinyImageNet**: cross-dataset validation of the four-regime
taxonomy. Same ViT-Base architecture as ImageNet-100, upscaled 64→224
(milder than CIFAR's 32→224), 200 classes, 100K train images.

**Output**: 12 checkpoints saved under
`/content/drive/MyDrive/Trained models_TinyImageNet/{pe_type}_seed{seed}/best_model.pth`

**Compute estimate**: ~19h NVIDIA RTX PRO 6000 Blackwell Server Edition per model 

⚠️ Colab sessions disconnect after ~24h. Strategy: train **one (pe_type, seed)
configuration per session**. The notebook has a `--resume` flag that skips
already-completed models so you can re-run the same cell safely.


## 1. Mount Google Drive

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')
print("Drive mounted.")


## 2. Configuration

Edit `CURRENT_PE_TYPE` and `CURRENT_SEED` between sessions to choose
which configuration to train.


In [ ]:
# ============================================================
# CONFIG — edit between sessions to pick which config to train
# ============================================================

# Where your existing full_scale_experiment.py lives (we import
# VisionTransformer from it). The ORIGINAL is fine — no patch needed
# for TinyImageNet (we use the 4 standard PE methods, not 2D-ALiBi).
ORIG_SCRIPT_PATH = "/content/drive/MyDrive/full_scale_experiment.py"

# Where the TinyImageNet training script lives in Drive
TRAIN_SCRIPT_PATH = "/content/drive/MyDrive/tinyimagenet_experiment.py"

# Working directory in Colab session
WORK_DIR = "/content/work"

# Where TinyImageNet dataset will live (downloaded once, ~250 MB)
# DATA_DIR = "/content/drive/MyDrive/tinyimagenet"
DATA_DIR = "/content/"

# Where checkpoints will be saved
OUTPUT_DIR = "/content/drive/MyDrive/Trained models_TinyImageNet"

# Which (PE, seed) configuration to train in THIS session
# Across 12 sessions you will cycle through:
#   ('learned', 42), ('learned', 123), ('learned', 456),
#   ('sinusoidal', 42), ... ('alibi', 456)
CURRENT_PE_TYPE = "sinusoidal"   # learned | sinusoidal | rope | alibi
CURRENT_SEED    = 42           # 42 | 123 | 456

# Training settings (match ImageNet-100 setup)
EPOCHS      = 300
BATCH_SIZE  = 128
NUM_WORKERS = 12

# Pilot run to verify pipeline before committing 10-12h
RUN_PILOT_FIRST = False   # set False after the first successful pilot
PILOT_EPOCHS    = 5

# Download dataset on first run? (After it's in Drive, set to False)
DOWNLOAD_IF_NEEDED = True

print(f"Original ViT script:  {ORIG_SCRIPT_PATH}")
print(f"Training script:      {TRAIN_SCRIPT_PATH}")
print(f"Data dir:             {DATA_DIR}")
print(f"Output dir:           {OUTPUT_DIR}")
print()
print(f"This session:         pe_type={CURRENT_PE_TYPE}, seed={CURRENT_SEED}")
print(f"Epochs:               {EPOCHS}")
print(f"Run pilot first?      {RUN_PILOT_FIRST} ({PILOT_EPOCHS} epochs)")


## 3. Verify GPU

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU! Switch runtime: Runtime > Change runtime type > GPU"
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"CUDA: {torch.version.cuda}")
print(f"PyTorch: {torch.__version__}")


## 4. Install dependencies

In [ ]:
!pip install -q timm tqdm scikit-learn scipy matplotlib
print("Dependencies installed.")


## 5. Verify required files

Before running this cell, upload TWO files to your Drive:

1. **`full_scale_experiment.py`** — your existing IN-100 training script
   (TinyImageNet script imports `VisionTransformer` from it)
2. **`tinyimagenet_experiment.py`** — the TinyImageNet training script
   I gave you

Recommended location: `/content/drive/MyDrive/` (root of your Drive).


In [ ]:
import os

required = [
    (ORIG_SCRIPT_PATH,  "full_scale_experiment.py"),
    (TRAIN_SCRIPT_PATH, "tinyimagenet_experiment.py"),
]
missing = []
for path, label in required:
    if not os.path.exists(path):
        missing.append(f"  ✗ {label}: {path}")
    else:
        print(f"  ✓ Found {label}: {path}")

if missing:
    print()
    print("MISSING FILES:")
    print("\n".join(missing))
    raise FileNotFoundError("Upload the missing files to Drive before continuing.")
else:
    print()
    print("All required files present.")


## 6. Stage scripts into working directory

The training script imports `VisionTransformer` from
`full_scale_experiment.py`, so both must be in the same directory on
the Colab filesystem (not Drive — imports through Drive can be flaky).


In [ ]:
import os
import shutil

os.makedirs(WORK_DIR, exist_ok=True)

# Copy both scripts into work dir
for src in [ORIG_SCRIPT_PATH, TRAIN_SCRIPT_PATH]:
    dst = os.path.join(WORK_DIR, os.path.basename(src))
    shutil.copy(src, dst)
    print(f"  Copied: {src} -> {dst}")

print(f"\nScripts ready in {WORK_DIR}/")


## 7. Download TinyImageNet (one-time)

Downloads the ~250 MB zip from Stanford's CS231n server and
reorganizes the val/ split into ImageFolder format. After the first
successful run, the data is on Drive and this becomes a no-op.


In [ ]:
import os

if DOWNLOAD_IF_NEEDED:
    cmd = (f"cd {WORK_DIR} && python tinyimagenet_experiment.py "
            f"--download_only --data_dir '{DATA_DIR}'")
    print(f"Running: {cmd}\n")
    exit_code = os.system(cmd)
    if exit_code != 0:
        raise RuntimeError(f"Download failed with exit code {exit_code}")
    print("\nDataset ready.")
else:
    print("Skipping download (DOWNLOAD_IF_NEEDED=False).")

# Sanity check
extracted = os.path.join(DATA_DIR, 'tiny-imagenet-200')
if os.path.isdir(extracted):
    n_train = len([d for d in os.listdir(os.path.join(extracted, 'train'))
                    if os.path.isdir(os.path.join(extracted, 'train', d))])
    val_reorg = os.path.join(extracted, 'val_reorganised')
    n_val = (len([d for d in os.listdir(val_reorg)
                   if os.path.isdir(os.path.join(val_reorg, d))])
             if os.path.isdir(val_reorg) else 0)
    print(f"\nVerified: {n_train} train classes, {n_val} val classes "
          f"(both should be 200)")


## 8. Pilot run (recommended for first time)

Runs the chosen (pe_type, seed) configuration for just 5 epochs.
Confirms:
- Both scripts import correctly
- TinyImageNet data loads with the right shapes
- Model trains without OOM
- Loss decreases epoch-over-epoch

Takes ~20-30 min. Skip after first successful pilot
(`RUN_PILOT_FIRST = False` in config cell).


In [ ]:
if RUN_PILOT_FIRST:
    pilot_output = "/content/pilot_test_tinyin"
    os.makedirs(pilot_output, exist_ok=True)

    cmd = f'''cd {WORK_DIR} && python tinyimagenet_experiment.py \
        --pe_type {CURRENT_PE_TYPE} \
        --seed {CURRENT_SEED} \
        --data_dir "{DATA_DIR}" \
        --output_dir "{pilot_output}" \
        --epochs {PILOT_EPOCHS} \
        --batch_size {BATCH_SIZE} \
        --num_workers {NUM_WORKERS}'''

    print(f"PILOT — pe_type={CURRENT_PE_TYPE}, seed={CURRENT_SEED}, "
          f"{PILOT_EPOCHS} epochs (~20-30 min)")
    print("=" * 60)
    print(cmd)
    print("=" * 60)
    print()

    exit_code = os.system(cmd)

    if exit_code != 0:
        raise RuntimeError(f"Pilot failed with exit code {exit_code}")

    print()
    print("✅ Pilot complete. Inspect loss curve above.")
    print("If loss decreased over epochs, proceed to the full run.")
    print()
    print("To skip pilot in future sessions, set RUN_PILOT_FIRST = False.")
else:
    print("Pilot skipped (RUN_PILOT_FIRST = False).")


In [ ]:
import os, json, time

out_dir = "/content/pilot_test_tinyin/learned_seed42"

print("=== Pilot output check ===\n")

# 1. Sadržaj output direktorijuma
if os.path.isdir(out_dir):
    for f in sorted(os.listdir(out_dir)):
        path = os.path.join(out_dir, f)
        size = os.path.getsize(path) / 1e6
        mtime = time.strftime('%H:%M:%S', time.localtime(os.path.getmtime(path)))
        print(f"  {f}: {size:.1f} MB, modified {mtime}")
else:
    print(f"  Output direktorijum ne postoji: {out_dir}")

# 2. Training history (KLJUČNA provera)
hist_path = os.path.join(out_dir, "training_history.json")
if os.path.isfile(hist_path):
    with open(hist_path) as f:
        h = json.load(f)
    print(f"\n=== Training history ===")
    print(f"  Epochs completed: {len(h['val_acc'])}")
    print(f"\n  Epoch  |  Train Loss  |  Val Acc  |  Time")
    print(f"  -------|-------------|-----------|--------")
    for i, (loss, acc, t) in enumerate(zip(h['train_loss'], h['val_acc'], h['epoch_time']), 1):
        print(f"   {i:3d}   |   {loss:6.3f}    |  {acc:5.2f}%   | {t:5.1f}s")

    # Trend analiza
    print(f"\n=== Trend analysis ===")
    if len(h['train_loss']) >= 2:
        loss_trend = h['train_loss'][-1] - h['train_loss'][0]
        acc_trend  = h['val_acc'][-1]  - h['val_acc'][0]
        print(f"  Train loss change: {loss_trend:+.3f} (should be negative)")
        print(f"  Val acc change:    {acc_trend:+.2f}pp (should be positive)")

        if loss_trend < 0 and acc_trend > 0:
            print(f"  ✅ Pilot looks healthy — proceed to full run")
        elif loss_trend < 0:
            print(f"  ⚠️  Loss decreasing but val acc not improving (5 epoch je malo)")
        else:
            print(f"  ❌ Loss not decreasing — something is wrong")
else:
    print("\n  ❌ training_history.json ne postoji — trening nije završio")

In [ ]:
import os
print(f"CPU cores: {os.cpu_count()}")

In [ ]:
import os, time

# Proveri output direktorijum
out_dir = "/content/pilot_test_tinyin/learned_seed42"

# 1. Da li postoji best_model.pth i koliko je velik
ckpt = os.path.join(out_dir, "best_model.pth")
if os.path.isfile(ckpt):
    size_mb = os.path.getsize(ckpt) / 1e6
    mtime = time.ctime(os.path.getmtime(ckpt))
    print(f"  ✓ best_model.pth: {size_mb:.0f} MB, last modified: {mtime}")
else:
    print(f"  ✗ best_model.pth ne postoji jos")

# 2. Da li postoji training_history.json (samo na kraju)
hist = os.path.join(out_dir, "training_history.json")
if os.path.isfile(hist):
    print(f"  ✓ training_history.json postoji — trening je gotov!")
    import json
    with open(hist) as f:
        h = json.load(f)
    print(f"     {len(h['val_acc'])} epoch(s) completed")
    print(f"     val_acc per epoch: {[f'{v:.2f}%' for v in h['val_acc']]}")
    print(f"     train_loss:        {[f'{v:.3f}' for v in h['train_loss']]}")
else:
    print(f"  ✗ training_history.json jos ne postoji — trening jos traje")

# 3. GPU usage
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=utilization.gpu,memory.used,memory.total',
                          '--format=csv,noheader,nounits'],
                         capture_output=True, text=True)
print(f"  GPU: {result.stdout.strip()}")

## 9. Full training run — current configuration

Trains a single (pe_type, seed) configuration for 300 epochs.
Expected wall time: ~10-12h H100.

Strategy across sessions (12 total runs):

| Session | pe_type    | seed |
|---------|------------|------|
| 1       | learned    | 42   |
| 2       | learned    | 123  |
| 3       | learned    | 456  |
| 4       | sinusoidal | 42   |
| ...     | ...        | ...  |
| 12      | alibi      | 456  |

The script has `--resume` logic: if `best_model.pth` already exists
for the chosen config, it skips training. So running this cell with
a config you've already trained is safe.


In [ ]:
cmd = f'''cd {WORK_DIR} && python tinyimagenet_experiment.py \
    --pe_type {CURRENT_PE_TYPE} \
    --seed {CURRENT_SEED} \
    --data_dir "{DATA_DIR}" \
    --output_dir "{OUTPUT_DIR}" \
    --epochs {EPOCHS} \
    --batch_size {BATCH_SIZE} \
    --num_workers {NUM_WORKERS} \
    --resume'''

print(f"FULL TRAINING — pe_type={CURRENT_PE_TYPE}, seed={CURRENT_SEED}, "
      f"{EPOCHS} epochs")
print("=" * 60)
print(cmd)
print("=" * 60)
print()
print(f"Expected wall time: ~10-12h H100")
print(f"Best checkpoint will be saved to:")
print(f"  {OUTPUT_DIR}/{CURRENT_PE_TYPE}_seed{CURRENT_SEED}/best_model.pth")
print()

import subprocess
import sys

process = subprocess.Popen(
    cmd, shell=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    bufsize=1,
    universal_newlines=True,
)

for line in process.stdout:
    print(line, end='', flush=True)
    sys.stdout.flush()

exit_code = process.wait()

if exit_code != 0:
    print(f"\n⚠️ Training exited with code {exit_code}")
    print("Check the output above. If Colab disconnected, the latest best")
    print("checkpoint should still be in Drive.")
else:
    print(f"\n✅ Training complete for {CURRENT_PE_TYPE}_seed{CURRENT_SEED}.")
    print(f"\nNext step: edit CURRENT_PE_TYPE / CURRENT_SEED in the config")
    print(f"cell and restart the runtime to train the next configuration.")


FULL TRAINING — pe_type=learned, seed=42, 300 epochs
cd /content/work && python tinyimagenet_experiment.py     --pe_type learned     --seed 42     --data_dir "/content/"     --output_dir "/content/drive/MyDrive/Trained models_TinyImageNet"     --epochs 300     --batch_size 128     --num_workers 12     --resume

Expected wall time: ~10-12h H100
Best checkpoint will be saved to:
  /content/drive/MyDrive/Trained models_TinyImageNet/learned_seed42/best_model.pth

Device: cuda
  GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
Config: {
  "img_size": 224,
  "patch_size": 16,
  "num_classes": 200,
  "embed_dim": 768,
  "depth": 12,
  "num_heads": 12,
  "mlp_ratio": 4.0,
  "dropout": 0.1,
  "epochs": 300,
  "batch_size": 128,
  "lr": 0.0003,
  "warmup_epochs": 20,
  "weight_decay": 0.1,
  "label_smoothing": 0.1,
  "mixup_alpha": 0.8
}
  TinyImageNet already prepared at /content/tiny-imagenet-200
Building data loaders ...
  Train batches: 781, Val batches: 79

>>> Training learned_seed42
    

KeyboardInterrupt: 

## 10. Status: which (pe_type, seed) configs are done

In [ ]:
import os

PE_LIST   = ['learned', 'sinusoidal', 'rope', 'alibi']
SEED_LIST = [42, 123, 456]

print("TinyImageNet training status")
print("=" * 60)
print(f"{'pe_type':12s} {'seed':>5s}    {'status':20s} {'size':>8s}")
print("-" * 60)

done_count = 0
for pe in PE_LIST:
    for seed in SEED_LIST:
        ckpt = os.path.join(OUTPUT_DIR, f"{pe}_seed{seed}", "best_model.pth")
        if os.path.isfile(ckpt):
            size_mb = os.path.getsize(ckpt) / 1e6
            print(f"{pe:12s} {seed:>5d}    ✓ done             {size_mb:>5.0f} MB")
            done_count += 1
        else:
            print(f"{pe:12s} {seed:>5d}    ✗ not trained          —")

print("-" * 60)
print(f"Total: {done_count}/12 configurations complete")
print()

if done_count == 12:
    print("✅ All 12 models trained. Ready for revision_analysis_v2.ipynb")
    print("   with PE_TYPES = ['learned', 'sinusoidal', 'rope', 'alibi']")
    print("   on the TinyImageNet checkpoints.")
elif done_count > 0:
    # Suggest next config
    for pe in PE_LIST:
        for seed in SEED_LIST:
            ckpt = os.path.join(OUTPUT_DIR, f"{pe}_seed{seed}", "best_model.pth")
            if not os.path.isfile(ckpt):
                print(f"Next to train:  CURRENT_PE_TYPE = '{pe}', CURRENT_SEED = {seed}")
                break
        else:
            continue
        break
